In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

All of the Exploratory Data Analysis was already done in lecture 02, so we are skipping it.

In [ ]:
def read_dataset() -> tuple[pd.DataFrame, pd.Series]:
    dataset = fetch_california_housing(as_frame=True)

    # Note:
    #   type: ignore is used to suppress type checker warnings.
    #   Ignore it yourself too, don't worry about it.
    X = dataset["data"]  # type: ignore
    y = dataset["target"]  # type: ignore

    return X.copy(), y.copy()


def cut_outliers(
    X: pd.DataFrame,
    y: pd.Series,
    lower_quantile: float = 0.01,
    upper_quantile: float = 0.99,
) -> tuple[pd.DataFrame, pd.Series]:
    """Cut outliers from the California Housing dataset.

    Parameters:

        X: pd.DataFrame
            Feature data.

        y: pd.Series
            Target data.

        lower_quantile: float
            Lower quantile threshold for outlier removal.

        upper_quantile: float
            Upper quantile threshold for outlier removal.

    Returns:

        X_clean: pd.DataFrame
            Feature data with outliers removed.

        y_clean: pd.Series
            Target data with outliers removed.
    """
    # Combine X and y into a single DataFrame for easier processing.
    data = X.copy()
    data["target"] = y

    # Create a boolean mask for rows that are within the bounds for all columns.
    mask = np.ones(len(data), dtype=bool)
    for column in data.columns:
        if column in ["Longitude", "Latitude"]:
            continue
        if column == "target":
            mask &= data["target"] < 5.0
        elif column == "HouseAge":
            mask &= data[column] <= 50.0
        elif column == "MedInc":
            mask &= data[column] < 15.0
        else:
            # Calculate the lower and upper bounds for each column.
            lower_bounds = data[column].quantile(lower_quantile)
            upper_bounds = data[column].quantile(upper_quantile)
            mask &= (data[column] >= lower_bounds) & (data[column] <= upper_bounds)

    # Apply the mask to filter the DataFrame.
    data_clean = data[mask]

    # Separate the cleaned DataFrame back into X and y.
    X_clean = data_clean.drop(columns=["target"])
    y_clean = data_clean["target"]

    return X_clean, y_clean


def compute_log(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """Apply logarithmic transformation to skewed features and target variable of
    the California Housing dataset.

    Parameters:

        X: pd.DataFrame
            Feature data.

        y: pd.Series
            Target data.

    Returns:

        X_log: pd.DataFrame
            Feature data with logarithmic transformation applied to skewed features.

        y_log: pd.Series
            Target data with logarithmic transformation applied.
    """
    X_log = X.copy()
    y_log = y.copy()

    # List of features to apply logarithmic transformation.
    skewed_features = ["MedInc", "AveRooms", "AveBedrms", "Population", "AveOccup"]

    for feature in skewed_features:
        X_log[feature] = X_log[feature].apply(np.log1p)

    # Apply logarithmic transformation to the target variable.
    y_log = y_log.apply(np.log1p)

    return X_log, y_log


def process_original_dataset(X, y):
    """Process the original California Housing dataset by removing outliers and
    applying logarithmic transformation to skewed features and target variable.

    Parameters:

        X: pd.DataFrame
            Original feature data.

        y: pd.Series
            Original target data.
    Returns:

        X_processed: pd.DataFrame
            Processed feature data.

        y_processed: pd.Series
            Processed target data.
    """
    X_clean, y_clean = cut_outliers(X, y)
    X_processed, y_processed = compute_log(X_clean, y_clean)
    return X_processed.copy(), y_processed.copy()

In [ ]:
X_original, y_original = read_dataset()
X, y = process_original_dataset(X_original, y_original)

## Train-test split 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

(
    X_train_model_selection,
    X_test_model_selection,
    y_train_model_selection,
    y_test_model_selection,
) = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
)

In [ ]:
print(f'Original dataset:\n{X.shape=}, {y.shape=}')
print()
print(f'Test set:\n{X_test.shape=}, {y_test.shape=}')
print(f'Training set:\n{X_train.shape=}, {y_train.shape=}')
print()
print(f'Validation set:\n{X_test_model_selection.shape=}, {y_test_model_selection.shape=}')
print(f'Model selection training set:\n{X_train_model_selection.shape=}, {y_train_model_selection.shape=}')